# Create directed narration with Claude and FlowSpeech

This notebook combines Claude's writing and instruction-following capabilities with [FlowSpeech](https://flowspeech.io/), a context-aware text-to-speech API. Claude turns a brief into a compact two-speaker script with delivery cues, and FlowSpeech converts that script into playable audio.

You will learn how to:

1. Ask Claude for structured dialogue suitable for speech synthesis.
2. Preserve emotion and pause directions in the generated script.
3. Assign a different FlowSpeech voice to each speaker.
4. Decode the API response into a WAV file and play it in Jupyter.

## Installation

Install the dependencies listed beside this notebook.

In [ ]:
%pip install -r requirements.txt

## Configure API keys

Copy `.env.example` to `.env`, then add your Anthropic and FlowSpeech API keys. The notebook reads both values from the environment and never prints them.

In [ ]:
import base64
import os
import wave
from pathlib import Path

import anthropic
import requests
from dotenv import load_dotenv
from IPython.display import Audio, display

load_dotenv()

anthropic_api_key = os.environ.get("ANTHROPIC_API_KEY")
flowspeech_api_key = os.environ.get("FLOWSPEECH_API_KEY")

assert anthropic_api_key, "Set ANTHROPIC_API_KEY in your environment."
assert flowspeech_api_key, "Set FLOWSPEECH_API_KEY in your environment."

## Generate a directed script with Claude

FlowSpeech accepts speaker labels and delivery cues in the text. The prompt below asks Claude to produce only the script so it can be sent directly to the speech API.

In [ ]:
client = anthropic.Anthropic(api_key=anthropic_api_key)

brief = (
    "Write a 20-second exchange between a museum guide and a curious visitor "
    "who have just discovered a hidden room behind a painting."
)

prompt = f"""
Create a short two-speaker narration from this brief:
{brief}

Requirements:
- Use exactly the speaker labels Guide and Visitor.
- Write 3 to 5 lines total.
- Put one natural delivery cue such as [quietly] or [excited] before selected lines.
- Include one [pause 0.6s] cue where a dramatic pause helps.
- Return only the script, with each line formatted as Speaker: dialogue.
"""

message = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=300,
    temperature=0.4,
    messages=[{"role": "user", "content": prompt}],
)

script = "".join(block.text for block in message.content if block.type == "text").strip()
print(script)

## Synthesize the script with FlowSpeech

The public FlowSpeech endpoint accepts the generated text and a list of speaker-to-voice assignments. The response includes base64-encoded audio plus metadata such as sample rate and channel count.

In [ ]:
response = requests.post(
    "https://flowspeech.io/api/ai/text-to-speech",
    headers={
        "Authorization": f"Bearer {flowspeech_api_key}",
        "Content-Type": "application/json",
    },
    json={
        "text": script,
        "originalText": script,
        "speakers": [
            {"speaker": "Guide", "voiceName": "Kore"},
            {"speaker": "Visitor", "voiceName": "Puck"},
        ],
    },
    timeout=80,
)
response.raise_for_status()
payload = response.json()

if payload.get("code") != 0:
    raise RuntimeError(payload.get("message", "FlowSpeech returned an error."))

audio_data = payload["data"]
audio_bytes = base64.b64decode(audio_data["audioBase64"])
output_path = Path("claude_flowspeech_demo.wav")

with wave.open(str(output_path), "wb") as wav_file:
    wav_file.setnchannels(audio_data.get("numChannels", 1))
    wav_file.setsampwidth(audio_data.get("bitsPerSample", 16) // 8)
    wav_file.setframerate(audio_data.get("sampleRate", 24000))
    wav_file.writeframes(audio_bytes)

print(
    f"Saved {len(audio_bytes):,} bytes to {output_path} "
    f"({audio_data.get('sampleRate', 'unknown')} Hz, "
    f"{audio_data.get('numChannels', 'unknown')} channel)."
)

In [ ]:
display(Audio(filename=str(output_path)))

## Production considerations

- Validate or constrain Claude's speaker labels before sending user-generated scripts to the TTS API.
- Keep API keys on the server and never expose them in browser code.
- Add retries for transient network errors, but do not retry invalid requests indefinitely.
- Cache completed audio when the script and voice assignments are unchanged.
- Obtain appropriate consent before synthesizing or distributing speech associated with a real person's identity.

From here, you can adapt the same pattern for narrated explainers, dialogue previews, accessible reading experiences, and voice-enabled Claude applications.